# CSoT'26 - ML in Astronomy - Week 2 . Part 2: Your First Neural Network (Starter)

**Goal:** Define a 2-layer fully-connected network (MLP) with `nn.Module`, forward-pass a real batch, set up a loss + optimiser, and run a single optimisation step to watch the loss drop.

**Before you begin:**
1. Switch this notebook to a **GPU runtime** (`Runtime -> Change runtime type -> GPU`).
2. Read [`04-building-models-with-nn-module.md`](../04-building-models-with-nn-module.md) and [`05-loss-functions-and-optimisers.md`](../05-loss-functions-and-optimisers.md).

Replace each `TODO` with working code. **Do not** open the solution until you've genuinely attempted every TODO. (We *set up* training here; the full multi-epoch training loop is Week 3.)

## Step 0 — Re-create the Week 1 data pipeline

Week 2 builds directly on the `DataLoader`s from Week 1. The cells below reproduce that pipeline (download is commented out — uncomment it the first time, exactly as in [`week1_data_solution.ipynb`](../../Week-1/notebooks/week1_data_solution.ipynb)). If you saved `galaxy_data/` to Google Drive in Week 1, just re-mount Drive and point `DATA_ROOT` at it instead of re-downloading.

After this section you should have `train_loader`, `val_loader`, `test_loader`, `train_ds`, and `num_classes`.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [ ]:
# TODO: paste your Week 1 data pipeline here so that the following names are defined:
#   train_ds, val_ds, test_ds, train_loader, val_loader, test_loader, num_classes
#
# The quickest path is to copy the data-prep cells from
# ../../Week-1/notebooks/week1_data_solution.ipynb (Steps 1-8), then add:
#   num_classes = len(train_ds.classes)

from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('username')
os.environ['KAGGLE_KEY'] = userdata.get('gal_morph')

RAW_ROOT = Path("galaxy_raw")      # <-- set up download here
IMAGES_DIR = RAW_ROOT / "images_gz2" /"images"  # flat image folder
DATA_ROOT = Path("galaxy_data")    # ImageFolder root (built in Step 3)

RAW_ROOT.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images -p {RAW_ROOT}
!unzip -q {RAW_ROOT}/galaxy-zoo-2-images.zip -d {RAW_ROOT}
print("Files extracted")

print("Downloading gz2_hart16.csv")
!wget -P {RAW_ROOT} https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz

print("Extracting labels file")
!gunzip -f {RAW_ROOT}/gz2_hart16.csv.gz

def high_level_label(gz2_class):
  if "E" in gz2_class:
    return "elliptical"
  elif "SB" in gz2_class:
    return "spiral_barred"
  elif "S" in gz2_class:
    return "spiral"
  else:
    return None

def build_imagefolder_layout(images_dir, mapping_csv,labels_csv, out_root, per_class=200, seed=42,
):
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv).rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    df = df.dropna(subset=["label"])

    images_dir = Path(images_dir)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    counts = {}
    for label in sorted(df["label"].unique()):
        class_dir = out_root / label
        class_dir.mkdir(exist_ok=True)
        rows = df[df["label"] == label]
        if len(rows) > per_class:
            rows = rows.sample(n=per_class, random_state=seed)
        linked = 0
        for _, row in rows.iterrows():
            src = images_dir / f"{int(row.asset_id)}.jpg"
            dst = class_dir / f"{int(row.asset_id)}.jpg"
            if src.exists() and not dst.exists():
                os.symlink(src.resolve(), dst)
                linked += 1
        counts[label] = linked
    return counts

PER_CLASS = 200
counts = build_imagefolder_layout(
    IMAGES_DIR,
    RAW_ROOT / "gz2_filename_mapping.csv",
    RAW_ROOT / "gz2_hart16.csv",
    DATA_ROOT,
    per_class=PER_CLASS,
)
print("Symlinked per class:", counts)
print("DATA_ROOT classes:", sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir()))

Dataset URL: https://www.kaggle.com/datasets/jaimetrickz/galaxy-zoo-2-images
License(s): Attribution 4.0 International (CC BY 4.0)
100% 3.06G/3.06G [01:25<00:00, 38.5MB/s]

Files extracted
--2026-06-09 14:54:25--  https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz
Resolving gz2hart.s3.amazonaws.com (gz2hart.s3.amazonaws.com)... 16.15.207.5, 52.217.118.41, 52.216.39.81, ...
Connecting to gz2hart.s3.amazonaws.com (gz2hart.s3.amazonaws.com)|16.15.207.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 78513011 (75M) [application/x-gzip]
Saving to: ‘galaxy_raw/gz2_hart16.csv.gz’

gz2_hart16.csv.gz   100%[===================>]  74.88M  21.9MB/s    in 3.4s    

2026-06-09 14:54:29 (21.9 MB/s) - ‘galaxy_raw/gz2_hart16.csv.gz’ saved [78513011/78513011]

Extracting labels file
Symlinked per class: {'elliptical': 200, 'spiral': 200, 'spiral_barred': 200}
DATA_ROOT classes: ['elliptical', 'spiral', 'spiral_barred']


In [ ]:
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5],
    ),
])

In [ ]:
dataset = ImageFolder(root=DATA_ROOT, transform=transform)
total_size = len(dataset)
train_size = int(0.70 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

num_classes = len(train_ds.dataset.classes)

## Step 1 - Define the model

A 2-layer MLP: flatten -> Linear(12288, 128) -> ReLU -> Linear(128, num_classes). The output layer returns **raw logits** (no softmax - `CrossEntropyLoss` adds it). Don't forget `super().__init__()`.

In [ ]:
# TODO: define GalaxyMLP(nn.Module) with:
#   self.flatten = nn.Flatten()
#   self.fc1 = nn.Linear(3*64*64, 128); self.relu = nn.ReLU()
#   self.fc2 = nn.Linear(128, num_classes)
# and a forward(self, x) that runs flatten -> fc1 -> relu -> fc2 and returns logits.
class GalaxyMLP(nn.Module):
    def __init__(self, in_features=3*64*64, hidden=128, num_classes=3):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(in_features, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


## Step 2 - Instantiate and move to the device

Use the real `num_classes` from your data, and `.to(device)` so the model lives where the batches will.

In [ ]:
# TODO: model = GalaxyMLP(num_classes=num_classes).to(device)
model = GalaxyMLP(num_classes=num_classes).to(device)

## Step 3 - Inspect the architecture

`print(model)` shows the layers; counting `.parameters()` shows how many weights there are. Notice that the first `Linear` (12288 x 128) dominates - a direct cost of flattening.

In [ ]:
# TODO: print(model), then print total and trainable parameter counts.
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

GalaxyMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=12288, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)
1573379


## Step 4 - Forward-pass one real batch

Pull a batch from `train_loader`, move it to the device, and run it through the model. The output should be logits of shape `(batch_size, num_classes)`.

In [ ]:
# TODO: images, labels = next(iter(train_loader)); move both to device.
#       logits = model(images); print logits.shape and confirm it's (B, num_classes).
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)
logits = model(images)
print(logits.shape)

torch.Size([32, 3])


## Step 5 - Loss and optimiser

`CrossEntropyLoss` consumes raw logits + integer labels. `Adam` with `lr=1e-3` is the sensible default.

lr had to be decreased for loss to decrease, 1e-3 was too aggressive

In [ ]:
# TODO: criterion = nn.CrossEntropyLoss()
#       optimizer = optim.Adam(model.parameters(), lr=1e-3)
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

## Step 6 - Sanity-check the starting loss

For an untrained model on `C` balanced classes the loss should sit near `ln(C)`. If it's wildly off (or `nan`), suspect label dtype, a stray softmax, or unnormalised inputs.

In [ ]:
# TODO: loss = criterion(logits, labels); print loss.item() and compare to math.log(num_classes).
import math
loss = criterion(logits, labels)
print(loss.item())
print(math.log(num_classes))

1.103229284286499
1.0986122886681098


## Step 7 - One optimisation step (learning, in miniature)

The three lines at the heart of every training loop: clear gradients, backprop, update. Re-evaluate the loss on the *same* batch afterwards - it should drop. (Week 3 repeats this over all batches for many epochs.)

In [ ]:
# TODO: model.train()
#       optimizer.zero_grad(); loss.backward(); optimizer.step()
#       recompute the loss on the SAME batch and confirm it decreased.
model.train()
optimizer.zero_grad()
loss.backward()
optimizer.step()

post_outputs = model(images)
new_loss = criterion(post_outputs, labels)

In [ ]:
print(new_loss.item())

1.0722782611846924


## Step 8 (stretch) - How big does the model get?

Re-build the MLP with different hidden widths and print the parameter count each time. The first `Linear` (in_features x hidden) dominates - flattening is expensive. A CNN (Week 3) does more with far fewer parameters by sharing weights across the image.

In [ ]:
# TODO (optional): rebuild the MLP with hidden widths 64/128/256/512 and print param counts.

## Reflection *(write 2-3 sentences each)*

1. Your untrained loss should have been near `ln(num_classes)`. Why is that the expected starting value?
2. After one step the loss dropped on that batch. Why is that *not yet* the same as 'the model is trained'?
3. Compare the MLP's parameter count to what you'd expect a CNN to need. Why does flattening make the first layer so large?

*1. cross entropy loss = -log(p) where p is the predicted prob for the class. since it is initialised by random weights, each class (after softmax) has p = 1/(num_classes), so ce_loss = -log(1/num_classes) = log(num_classes)*<br>
*2. probably taking into account memorisation and overfitting.*<br>
*3. after flattening, every pixel becomes a feature and has to be connected to every other neuron in the next layer, which causes the first layer to explode. CNN doesn't flatten the image upfront, hence the number of features are considerably less.*